# scDMC Quick Start Tutorial

Welcome to the scDMC quick start guide! This tutorial provides step-by-step instructions on how to leverage the pre-trained scDMC foundation model for downstream single-cell analysis.

**Here is what we will cover:**
1. **Downloading & Loading Pre-trained Weights**: Fetching the pre-trained weights from HuggingFace.
2. **Data Preparation**: Converting custom `.h5ad` datasets into scDMC's unique dual-stream input format (Rank and EPE sequences).
3. **Low-Cost Fine-Tuning**: Adapting the pre-trained model for downstream classification tasks (e.g., cell type annotation) efficiently.

## 1. Downloading and Loading scDMC Pre-trained Weights
scDMC's pre-trained weights are hosted on HuggingFace for seamless integration. You can download and load them using the Hugging Face `huggingface_hub` Python package.

In [ ]:
import os
from huggingface_hub import snapshot_download

model_repo_id = "/scDMC-models"
local_pretrained_dir = "../models/pretrained_models/scDMC"
os.makedirs(local_pretrained_dir, exist_ok=True)

print(f"Downloading scDMC weights from {model_repo_id}...")
# Uncomment the code underneath to download the actual model:
# snapshot_download(
#     repo_id=model_repo_id, 
#     local_dir=local_pretrained_dir, 
#     allow_patterns=["*.json", "*.bin", "*.safetensors"]
# )
print("Weights are locally positioned at:", local_pretrained_dir)

## 2. Converting Custom `.h5ad` Data to Input Format
To use scDMC, your raw single-cell `.h5ad` data must be transformed into dual-stream components:
- **Rank**: Highlighting relative importance of expressed genes.
- **EPE (Expression Perception Enhanced)**: Contextualizing absolute expression levels.

In [ ]:
# The tokenization process uses the tokenizer scripts in the scDMC repository.
# You can run them via the command line or use a system call here.
import subprocess

# Assume you have your custom dataset: custom_data.h5ad
input_adata_path = "path/to/custom_data.h5ad"
output_dir = "./processed_data"

command = f"""\
python ../tokenizer/tokenizer_combine.py \
    --input_directory {input_adata_path} \
    --output_pathectory {output_dir} \
    --output_prefix my_custom_tokens
"""

print("Command to process your custom data:\n")
print(command)

# Uncomment to execute the tokenization task:
# subprocess.run(command, shell=True)

## 3. Low-cost Fine-Tuning for Downstream Tasks
To achieve optimal accuracy on specific tasks (e.g., classifying cell types), fine-tune the model representations alongside a fresh classification head.

In [ ]:
# To execute the fine tuning, we recommend utilizing the provided script.
command_finetune = f"""\
python ../downstream_tasks/finetune.py \
    --pretrained_model_path {local_pretrained_dir} \
    --finetune_data_path {output_dir}/my_custom_tokens.parquet \
    --output_path ./scDMC_finetuned_model \
    --epochs 5 \
    --learning_rate 1e-4 \
    --batch_size 16
"""
print("Command to run fine-tuning:\n")
print(command_finetune)

# Upon completion, the task-specific weights will be located in the --output_path
print("Fine-tuning complete!")